# How the visual search model works, step by step

This notebook walks through the priority-field model of visual search
fitted in this repository — the model that predicts **where the eyes go
next** during search. No advanced math or machine-learning background
needed: every step is shown with a figure.

**The big picture.** On each eye-movement decision, the model receives:

1. **A picture of the display** (raw pixels — nothing is labeled),
2. **The goal** ("find the green diamond") — just a *color direction*
   and a *shape template*, not the answer,
3. **Where the eyes currently are**, and
4. **Its own memory** of where targets and distractors appeared on
   previous trials (which it builds itself).

Its output is **a probability for every item on the screen** — "there
is a 55% chance the next eye movement lands on item 2." A single
predicted saccade is a random draw from those probabilities.

Requirements: run from the `Visual-Search/` folder; needs `numpy`,
`pandas`, `matplotlib`. All model weights are the fitted values (from
217,595 real human eye movements) — nothing is hand-tuned here.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

from front_end import render, form_map, COLORS, IMG
from build_contexts_v21 import (opponency_contrast, template_axis,
                                wedge_profiles, item_positions, NBINS, MAXR)

sigmoid = lambda x: 1.0 / (1.0 + np.exp(-x))
radii = np.linspace(0.09, MAXR, NBINS)      # distances along each ray

# the fitted weights (final model; see RESULTS.md for how they were fit)
fit = json.load(open("results_history_order.json"))["history_inside_window"]
W, R0 = fit["params"], fit["r0"]
for name, meaning in [
        ("g_T", "how strongly the goal color attracts"),
        ("w_p", "how strongly any object attracts (default path)"),
        ("g_form", "how strongly the goal shape attracts"),
        ("k", "attention window: how fast weighting falls off"),
        ("beta_T", "memory pull toward past TARGET locations"),
        ("beta_D", "memory push away from past DISTRACTOR locations"),
        ("eta_T", "how fast target memory updates (per trial)"),
        ("eta_D", "how fast distractor memory updates"),
        ("g_I", "penalty on items already inspected this trial")]:
    print(f"{name:8} = {W[name]:+7.3f}   {meaning}")

## Step 1 — The input: a picture

We build one search display of the kind used in the experiments: six
items on a ring, all green, except one **red color singleton** (the
distractor to be ignored). The **target** is the green **diamond**;
everything else is a circle. The participant starts fixating the
center.

This image — the pixels — is the model's entire stimulus input. The
model is never told which item is the target or the singleton.

In [ ]:
TARG, SING = 1, 4                      # slot indices (0..5)
pos, _ = item_positions(6)
items = []
for j in range(6):
    items.append(dict(x=pos[j][0], y=pos[j][1],
                      color="red" if j == SING else "green",
                      shape="diamond" if j == TARG else "circle"))
img = render(items)

plt.figure(figsize=(4, 4))
plt.imshow(img, origin="lower")
plt.plot(IMG/2, IMG/2, "k+", ms=12)
plt.title("the model's input: raw pixels\n(+ marks current fixation)")
plt.xticks([]); plt.yticks([])
plt.show()

## Step 2 — Early vision: contrast maps

The first processing stage mimics early visual cortex (following the
classic Itti & Koch salience model). From the pixels it computes
**color-opponency maps** — at every location, "how red-vs-green is
this spot compared to its neighborhood?" — using center-surround
filters at several spatial scales.

The key point: this stage is *fixed*. It has no idea what the task
is. It just measures local color contrast, everywhere, in parallel.

In [ ]:
maps = opponency_contrast(img)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(maps["RG"], origin="lower", cmap="RdBu_r",
               vmin=-np.abs(maps["RG"]).max(), vmax=np.abs(maps["RG"]).max())
axes[0].set_title("red-green opponency contrast\n(red spots +, green spots -)")
axes[1].imshow(maps["P"], origin="lower", cmap="magma")
axes[1].set_title("intensity contrast\n('something is here')")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

## Step 3 — The goal picks a direction (not an answer)

How does "find the *green* diamond" enter? Not as a label on any item.
The goal supplies a **direction in color space**: the model rotates
its opponency axes so that one axis points toward the target color.
Along that rotated axis, green things score **positive** and red
things score **negative** — automatically, from the pixels.

Notice what this means for ignoring the distractor: there is no
"suppress red" instruction anywhere. Red simply scores *negative* on
the attend-green axis, and (as we'll see in step 5) negative scores
get cut off. Ignoring is a by-product of attending — the model calls
this *relegation*.

In [ ]:
u = template_axis("green")             # unit vector toward 'green'
D_T = u[0] * maps["RG"] + u[1] * maps["BY"]

plt.figure(figsize=(4.5, 4))
v = np.abs(D_T).max()
plt.imshow(D_T, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
plt.title("template-aligned contrast\nblue = matches goal color, red = opposite")
plt.xticks([]); plt.yticks([])
plt.show()

## Step 4 — A visual LiDAR: sampling the world from the eyes

The maps live in screen coordinates, but decisions are made *from the
current fixation*. So the model samples every map along **rays fanning
out from where the eyes are** — like a LiDAR scanner mounted on the
eyeball. For each item we get a *radial profile*: what the sensory
evidence looks like along the ray pointing at that item, bin by bin
with distance.

Below: the goal-color profile toward three items. The plain green item
rises positive, the red singleton dips negative, and the target looks
like a plain green item here (its diamond *shape* is a separate
channel).

In [ ]:
prof, rng_read = wedge_profiles(maps, u, (0.0, 0.0), 6)
# standardize channels (as in the fits) so the weights apply
prof = prof / prof.std(axis=(0, 1), keepdims=True).clip(1e-9)

plt.figure(figsize=(7, 4))
for j, lab, c in [(TARG, "target (green diamond)", "#118844"),
                  (SING, "singleton (red)", "#aa2222"),
                  (0, "plain green circle", "#888888")]:
    plt.plot(radii, prof[j, :, 0], color=c, label=lab)
plt.axhline(0, color="k", lw=0.5)
plt.xlabel("distance from fixation (display units)")
plt.ylabel("goal-color evidence along the ray")
plt.legend(); plt.title("the visual LiDAR: radial evidence profiles")
plt.show()

## Step 5 — Weight, cut off, and window: the goal-modified salience map

Now the fitted weights act. Each ray bin gets a weighted sum of its
channels (goal color x g_T, plain presence x w_p), then **negative
values are cut to zero** (the rectification — this is where the red
singleton gets relegated), and the result is summed along the ray with
an **attention window**: a fitted sigmoid that weights near evidence
more than far evidence. Finally the shape channel adds the target's
diamond bonus.

The result is one number per item: its **stimulus priority**. Look at
the bar chart — the target is highest (color + shape), and the
singleton ends up *below* the plain items, with no suppression command
anywhere in the model.

In [ ]:
window = sigmoid(W["k"] * (R0 - radii))          # fitted attention window

mix = W["g_T"] * prof[..., 0] + W["g_O"] * prof[..., 1] + W["w_p"] * prof[..., 2]
stim = (np.maximum(mix, 0) * window).sum(axis=1)  # rectify, then window-sum

fmap = form_map(items)
to_px = lambda v: (v + 0.75) / 1.5 * IMG
FORM = np.array([fmap[int(to_px(pos[j][1])), int(to_px(pos[j][0]))]
                 for j in range(6)])
FORM = FORM / FORM.std().clip(1e-9)
stim = stim + W["g_form"] * FORM

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(radii, window)
axes[0].set_title("fitted attention window\n(weight vs distance from fixation)")
axes[0].set_xlabel("distance"); axes[0].set_ylabel("weight")
cols = ["#888888"] * 6
cols[TARG], cols[SING] = "#118844", "#aa2222"
axes[1].bar(range(1, 7), stim, color=cols)
axes[1].set_title("stimulus priority per item\n(green=target, red=singleton)")
axes[1].set_xlabel("item"); axes[1].set_ylabel("priority")
plt.show()

## Step 6 — Memory: selection history

The last ingredient is not in the image at all. The model keeps two
simple memories, updated after every trial:

- **h_T**: where have *targets* been lately?  (updates fast:
  eta_T = 0.61 means an event is mostly forgotten ~2 trials later)
- **h_D**: where have *singletons* been lately?  (updates ~3x slower)

The update rule is one line: `h = (1 - eta) * h;  h[location] += eta`.
Below we simulate 30 trials where the singleton loves one location:
watch the two memories build at their different speeds.

In [ ]:
rng = np.random.default_rng(0)
hT, hD = np.zeros(6), np.zeros(6)
hist_T, hist_D = [], []
for t in range(30):
    targ = rng.integers(6)                          # target: random slot
    sing = SING if rng.random() < 0.7 else rng.integers(6)   # biased
    hT *= (1 - W["eta_T"]); hT[targ] += W["eta_T"]
    hD *= (1 - W["eta_D"]); hD[sing] += W["eta_D"]
    hist_T.append(hT.copy()); hist_D.append(hD.copy())
hist_T, hist_D = np.array(hist_T), np.array(hist_D)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharex=True)
axes[0].plot(hist_T[:, SING], label=f"slot {SING+1}")
axes[0].plot(hist_T.max(axis=1), ls=":", label="whichever slot is hottest")
axes[0].set_title("target memory h_T: fast, jumpy")
axes[1].plot(hist_D[:, SING], label=f"slot {SING+1} (singleton's favorite)")
axes[1].set_title("distractor memory h_D: slow, steady build")
for ax in axes:
    ax.set_xlabel("trial"); ax.legend()
plt.show()

## Step 7 — Putting it together: the output

The final field for each item is:

    F = stimulus priority
        + window(distance) * ( beta_T * h_T  +  beta_D * h_D
                               + g_I * already_visited )

and the **output** is a softmax over F: a probability for each item.
Note the signs: target memory *adds* (beta_T > 0), singleton memory
*subtracts* (beta_D < 0), visited items are strongly penalized
(g_I = -6.3, the "inhibition of return").

Below, the model's actual output for our display, given the memory
state after those 30 biased trials — and 20 sampled saccades.

In [ ]:
win_item = sigmoid(W["k"] * (R0 - 0.5))     # window value at the ring
visited = np.zeros(6)
F = stim + win_item * (W["beta_T"] * hT + W["beta_D"] * hD
                       + W["g_I"] * visited)
p = np.exp(F - F.max()); p = p / p.sum()

plt.figure(figsize=(6, 3.5))
plt.bar(range(1, 7), 100 * p, color=cols)
plt.ylabel("P(first saccade lands here)  [%]")
plt.xlabel("item")
plt.title("THE OUTPUT: a probability for every item")
plt.show()

draws = rng.choice(6, size=20, p=p) + 1
print("20 sampled saccades (item numbers):", list(draws))
print(f"target = item {TARG+1}, singleton = item {SING+1}")

## Step 8 — A whole trial: the scanpath

If the first saccade misses the target, the model keeps going — and
two things change with every landing: (1) the visual LiDAR re-centers
on the new fixation (near items now dominate), and (2) the landed item
joins the *visited* list and is strongly avoided (g_I). The loop stops
when the target is fixated.

In [ ]:
fix, fixslot = (0.0, 0.0), None
visited = np.zeros(6)
path = [fix]
for step in range(4):
    pr, _ = wedge_profiles(maps, u, fix, 6)
    pr = pr / pr.std(axis=(0, 1), keepdims=True).clip(1e-9)
    mix = W["g_T"]*pr[..., 0] + W["g_O"]*pr[..., 1] + W["w_p"]*pr[..., 2]
    s = (np.maximum(mix, 0) * window).sum(axis=1) + W["g_form"] * FORM
    d = np.array([np.hypot(pos[j][0]-fix[0], pos[j][1]-fix[1]) for j in range(6)])
    wi = sigmoid(W["k"] * (R0 - d))
    F = s + wi * (W["beta_T"]*hT + W["beta_D"]*hD + W["g_I"]*visited)
    if fixslot is not None:
        F[fixslot] = -1e9                       # can't 're-choose' where you are
    p = np.exp(F - F.max()); p /= p.sum()
    choice = int(rng.choice(6, p=p))
    if fixslot is not None:
        visited[fixslot] = 1
    fixslot, fix = choice, pos[choice]
    path.append(fix)
    if choice == TARG:
        break

plt.figure(figsize=(4.5, 4.5))
plt.imshow(img, origin="lower", extent=[-0.75, 0.75, -0.75, 0.75])
xs, ys = zip(*path)
plt.plot(xs, ys, "o-", color="k", lw=2, ms=5)
plt.plot(xs[0], ys[0], "k+", ms=14)
plt.title(f"a sampled scanpath ({len(path)-1} saccades to the target)")
plt.xticks([]); plt.yticks([])
plt.show()

## Step 9 — Why this is a *psychological* model: an effect emerges

Nothing in the code above mentions "priming." Yet run the model
through a sequence of trials and a classic human effect appears by
itself: the model looks at the singleton *less* when it appears at
the same location as on the previous trial — because the slow
distractor memory is still parked there. Real humans: about 3% vs 8%
in this dataset; the model, below, produces the same pattern.

In [ ]:
# precompute stimulus priority for every (target, singleton) arrangement
lut = {}
for tg in range(6):
    for sg in range(6):
        if sg == tg:
            continue
        its = [dict(x=pos[j][0], y=pos[j][1],
                    color="red" if j == sg else "green",
                    shape="diamond" if j == tg else "circle")
               for j in range(6)]
        im = render(its)
        mp = opponency_contrast(im)
        pr, _ = wedge_profiles(mp, u, (0.0, 0.0), 6)
        pr = pr / pr.std(axis=(0, 1), keepdims=True).clip(1e-9)
        mix = W["g_T"]*pr[..., 0] + W["g_O"]*pr[..., 1] + W["w_p"]*pr[..., 2]
        fm = form_map(its)
        Fo = np.array([fm[int(to_px(pos[j][1])), int(to_px(pos[j][0]))]
                       for j in range(6)])
        lut[(tg, sg)] = ((np.maximum(mix, 0) * window).sum(1)
                         + W["g_form"] * Fo / Fo.std().clip(1e-9))

hT, hD = np.zeros(6), np.zeros(6)
prev_sing, rep, chg = None, [], []
for t in range(2000):
    tg = int(rng.integers(6))
    sg = int(rng.choice([j for j in range(6) if j != tg]))
    F = lut[(tg, sg)] + win_item * (W["beta_T"]*hT + W["beta_D"]*hD)
    p = np.exp(F - F.max()); p /= p.sum()
    (rep if sg == prev_sing else chg).append(p[sg])
    prev_sing = sg
    hT *= (1 - W["eta_T"]); hT[tg] += W["eta_T"]
    hD *= (1 - W["eta_D"]); hD[sg] += W["eta_D"]

plt.figure(figsize=(4.5, 3.5))
plt.bar(["singleton location\nREPEATED", "singleton location\nchanged"],
        [100*np.mean(rep), 100*np.mean(chg)], color=["#aa2222", "#dd9999"])
plt.ylabel("P(saccade to singleton) [%]")
plt.title("an emergent effect: location priming of suppression")
plt.show()

## Recap

| You give the model | It computes | It returns |
| --- | --- | --- |
| a picture (pixels) | contrast maps -> goal-rotated evidence -> rectified, window-weighted priority | |
| a goal (color direction + shape) | the weighting of that evidence | a **probability for every item** |
| the fixation position | where the rays are sampled from | (a saccade = one random draw) |
| the trial history | two leaky memories, added with signs | |

**How the ten weights were chosen:** by maximum likelihood — replay
217,595 real eye movements from 333 people, and adjust the weights so
the model assigns the highest possible probability to the item each
person actually looked at. On people the model never saw, it assigns
29% probability on average to the true choice (chance: 18%), names the
exact item first 54% of the time, and reproduces the classic findings
of this literature (distractor suppression, its persistence across
saccades, and both location-priming effects). See `RESULTS.md` for the
full record, and `docs/priority_field_visual_search_model.md` for the
theory.